# Meta Embedding

# Hate Data Cluster

## Cleaning the Data

In [ ]:
import pandas as pd
import re
from google.colab import files

# Define the cleaning function
def clean_hindi_text(text):
    # Define a regular expression pattern to match Hindi characters and Devanagari numbers
    hindi_pattern = re.compile(r'[ऀ-ॿ\s]+')
    devanagari_number_pattern = re.compile(r'[०-९]+')
    # Use the pattern to filter out non-Hindi characters
    cleaned_text = ' '.join(hindi_pattern.findall(text)) + ' '.join(devanagari_number_pattern.findall(text))
    return cleaned_text

# Load the dataset from your .xlsx file
input_file = "hasoc_2020_hi_train.xlsx"
df = pd.read_excel(input_file)

# Apply the cleaning function to a specific column in your DataFrame
column_to_clean = "text"  # Replace with the actual column name
df['Cleaned_Column'] = df[column_to_clean].apply(clean_hindi_text)

# Save the processed data to a new Excel file
output_file = "cleaned_dataset_2020_hasoc.xlsx"
df.to_excel(output_file, index=False)

# Download the cleaned dataset
files.download(output_file)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import re
from google.colab import files

# Define the cleaning function
def clean_hindi_text(text):
    # Define a regular expression pattern to match Hindi characters and Devanagari numbers
    hindi_pattern = re.compile(r'[ऀ-ॿ\s]+')
    devanagari_number_pattern = re.compile(r'[०-९]+')
    # Use the pattern to filter out non-Hindi characters
    cleaned_text = ' '.join(hindi_pattern.findall(text)) + ' '.join(devanagari_number_pattern.findall(text))
    return cleaned_text

# Load the dataset from your .xlsx file
input_file = "hi_Hasoc2021_train.csv"
df = pd.read_csv(input_file)

# Apply the cleaning function to a specific column in your DataFrame
column_to_clean = "text"  # Replace with the actual column name
df['Cleaned_Column'] = df[column_to_clean].apply(clean_hindi_text)

# Save the processed data to a new Excel file
output_file = "cleaned_dataset_2021_hasoc.xlsx"
df.to_excel(output_file, index=False)

# Download the cleaned dataset
files.download(output_file)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Concatenating the Data

In [ ]:
import pandas as pd
from google.colab import files

df1 = pd.read_excel('cleaned_dataset_2020_hasoc.xlsx')
df2 = pd.read_excel('cleaned_dataset_2021_hasoc.xlsx')

# Concatenate along columns and select specific columns
result = pd.concat([df1[['text', 'task_1', 'task_2', 'Cleaned_Column']],
                    df2[['text', 'task_1', 'task_2', 'Cleaned_Column']]], axis=0)

# If you want to reset the index
result = result.reset_index(drop=True)

# Save the result to a new Excel file
result.to_excel('hasoc_2020_and_2021.xlsx', index=False)

print("Concatenated DataFrame saved successfully.")

Concatenated DataFrame saved successfully.


In [ ]:
files.download('hasoc_2020_and_2021.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Extracting Embeddings from mBERT for Hate Data

In [ ]:
import pandas as pd
# Load your dataset
df = pd.read_excel('hasoc_2020_and_2021.xlsx')

# Filter data with 'Hate' label
hate_data = df[df['task_1'] == 'HOF']

non_hate = df[df['task_1'] != 'HOF']

print(len(hate_data))
print(len(non_hate))

2280
5277


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np


# Load your dataset
df = pd.read_excel('hasoc_2020_and_2021.xlsx')

# Filter data with 'Hate' label
hate_data = df[df['task_1'] == 'HOF']

# Initialize mBERT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
model = AutoModel.from_pretrained("bert-base-multilingual-cased")

# Tokenize the input texts
tokenized_data = hate_data['Cleaned_Column'].apply(lambda x: tokenizer(x, padding=True, truncation=True, return_tensors='pt'))

# Extract embeddings
embeddings = []
for input_ids in tokenized_data:
    with torch.no_grad():
        outputs = model(**input_ids)

    hidden_states = outputs.last_hidden_state
    embeddings.append(hidden_states[:, 0, :].numpy())


In [ ]:
# Save the embeddings as a PyTorch tensor
embeddings = torch.tensor(embeddings)

# Save the embeddings to a file named 'embeddings.pt'
torch.save(embeddings, 'Hasoc-2020_and_2021_Hate_Data_embeddings.pt')

## Finding K value

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import torch

# Load embeddings from the saved file
embeddings = torch.load('Hasoc-2020_and_2021_Hate_Data_embeddings.pt')

# Check if embeddings are available before attempting to stack
if embeddings is not None and embeddings.numel() > 0:
    # Flatten the embeddings to a 2D array
    embeddings = np.vstack(embeddings.numpy())  # Convert to NumPy array

    # Determine the optimal k value using the elbow method
    inertia = []
    k_values = range(1, 11)

    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=0)
        kmeans.fit(embeddings)
        inertia.append(kmeans.inertia_)

    # Plot the elbow curve to find the optimal k
    plt.figure(figsize=(8, 6))
    plt.plot(k_values, inertia, marker='o')
    plt.title('Elbow Method for Optimal k (Hasoc Hate Data)')
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('Inertia')
    plt.grid()
    plt.show()

print()


## Cluster

In [ ]:
import torch
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from scipy.stats import norm

# Load your embeddings from the file
embeddings = torch.load('Hasoc-2020_and_2021_Hate_Data_embeddings.pt')

# Convert the embeddings to a NumPy array and flatten it
embeddings = embeddings.numpy().reshape(embeddings.shape[0], -1)

# Create a KMeans instance with k=1
kmeans = KMeans(n_clusters=1)

# Fit the KMeans model to your data
kmeans.fit(embeddings)

# Get the cluster assignments for each data point
labels = kmeans.labels_

# Get the cluster center
cluster_center = kmeans.cluster_centers_[0]

# Calculate distances of each point to the cluster center
distances = np.linalg.norm(embeddings - cluster_center, axis=1)

# Fit a normal distribution to the distances
mu, sigma = norm.fit(distances)

# Define a threshold based on the normal distribution (e.g., mu ± 3 * sigma)
distance_threshold = 3 * sigma

# Remove outliers
filtered_embeddings = embeddings[distances <= mu + distance_threshold]

# Visualize the filtered cluster
plt.scatter(filtered_embeddings[:, 0], filtered_embeddings[:, 1], label='Cluster')

# Mark the cluster center with a different representation
plt.scatter(cluster_center[0], cluster_center[1], c='red', marker='x', s=100, label='Cluster Center')

# Set labels and legend
plt.title("Hate Data Cluster for Hasoc-2020 and 2021")
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend(loc='best')

# Show the plot
plt.show()

print("==============================================")
print("Cluster Centroid : ",sum(cluster_center)/len(cluster_center))
print("Cluster Centroid Dimentions : ",cluster_center.shape)
print("==============================================")


## Distance Calculating for the Hindi Hate Data

In [ ]:
import torch
import numpy as np
from sklearn.cluster import KMeans
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
from scipy.stats import norm

# Load the new dataset
import pandas as pd

# Assuming you have a CSV file with the dataset
df = pd.read_excel('ML_Project(With English Text).xlsx')

# Check for records where Hate is 1
hate_records = df[df['Hate'] == 1]

# Load BERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
model = AutoModel.from_pretrained("bert-base-multilingual-cased")

# Create a new column for Hindi embeddings
df['hin_dis'] = 0.0

# Calculate embeddings for Hindi text and measure distance
for index, row in hate_records.iterrows():
    hindi_text = row['Cleaned_Column']

    # Tokenize and get embeddings
    inputs = tokenizer(hindi_text, return_tensors="pt", truncation=True)
    outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()

    # Measure Euclidean distance
    distance = np.linalg.norm(embeddings - cluster_center)

    # Update the 'hin_dis' column
    df.at[index, 'hin_dis'] = distance

# Show the updated DataFrame
print(df[['Cleaned_Column', 'Hate', 'hin_dis']])

df.to_excel('output_file.xlsx', index=False)



## Extracting Embeddings form mBERT for Non-Hate Data

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np


# Load your dataset
df = pd.read_excel('hasoc_2020_and_2021.xlsx')

# Filter data with 'Hate' label
hate_data = df[df['task_1'] != 'HOF']

# Initialize mBERT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
model = AutoModel.from_pretrained("bert-base-multilingual-cased")

# Tokenize the input texts
tokenized_data = hate_data['Cleaned_Column'].apply(lambda x: tokenizer(x, padding=True, truncation=True, return_tensors='pt'))

# Extract embeddings
embeddings = []
for input_ids in tokenized_data:
    with torch.no_grad():
        outputs = model(**input_ids)

    hidden_states = outputs.last_hidden_state
    embeddings.append(hidden_states[:, 0, :].numpy())


In [ ]:
# Save the embeddings as a PyTorch tensor
embeddings = torch.tensor(embeddings)

# Save the embeddings to a file named 'embeddings.pt'
torch.save(embeddings, 'Hasoc-2020_and_2021_Non-Hate_Data_embeddings.pt')

## Finding K Value for Hindi Non-Hate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import torch

# Load embeddings from the saved file
embeddings = torch.load('Hasoc-2020_and_2021_Non-Hate_Data_embeddings.pt')

# Check if embeddings are available before attempting to stack
if embeddings is not None and embeddings.numel() > 0:
    # Flatten the embeddings to a 2D array
    embeddings = np.vstack(embeddings.numpy())  # Convert to NumPy array

    # Determine the optimal k value using the elbow method
    inertia = []
    k_values = range(1, 11)

    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=0)
        kmeans.fit(embeddings)
        inertia.append(kmeans.inertia_)

    # Plot the elbow curve to find the optimal k
    plt.figure(figsize=(8, 6))
    plt.plot(k_values, inertia, marker='o')
    plt.title('Elbow Method for Optimal k (for Non-Hate Data)')
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('Inertia')
    plt.grid()
    plt.show()

print()


## Cluster

In [ ]:
import torch
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from scipy.stats import norm

# Load your embeddings from the file
embeddings = torch.load('Hasoc-2020_and_2021_Non-Hate_Data_embeddings.pt')

# Convert the embeddings to a NumPy array and flatten it
embeddings = embeddings.numpy().reshape(embeddings.shape[0], -1)

# Create a KMeans instance with k=1
kmeans = KMeans(n_clusters=1)

# Fit the KMeans model to your data
kmeans.fit(embeddings)

# Get the cluster assignments for each data point
labels = kmeans.labels_

# Get the cluster center
cluster_center = kmeans.cluster_centers_[0]

# Calculate distances of each point to the cluster center
distances = np.linalg.norm(embeddings - cluster_center, axis=1)

# Fit a normal distribution to the distances
mu, sigma = norm.fit(distances)

# Define a threshold based on the normal distribution (e.g., mu ± 3 * sigma)
distance_threshold = 3 * sigma

# Remove outliers
filtered_embeddings = embeddings[distances <= mu + distance_threshold]

# Visualize the filtered cluster
plt.scatter(filtered_embeddings[:, 0], filtered_embeddings[:, 1], label='Cluster')

# Mark the cluster center with a different representation
plt.scatter(cluster_center[0], cluster_center[1], c='red', marker='x', s=100, label='Cluster Center')

# Set labels and legend
plt.title("Cluster for Non-Hate Data Hasoc 2020 and 2021")
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend(loc='best')

# Show the plot
plt.show()

print("==============================================")
print("Cluster Centroid : ",sum(cluster_center)/len(cluster_center))
print("Cluster Centroid Dimentions : ",cluster_center.shape)
print("==============================================")


## Distance Calculating for Hindi Non-Hate Data

In [ ]:
import torch
import numpy as np
from sklearn.cluster import KMeans
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
from scipy.stats import norm

# Load the new dataset
import pandas as pd

# Assuming you have a CSV file with the dataset
df = pd.read_excel('output_file_Dis_hate.xlsx')

# Check for records where Hate is 0
hate_records = df[df['Hate'] == 0]

# Load BERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
model = AutoModel.from_pretrained("bert-base-multilingual-cased")

# Create a new column for Hindi embeddings
df['hin_dis_non_hate'] = 0.0

# Calculate embeddings for Hindi text and measure distance
for index, row in hate_records.iterrows():
    hindi_text = row['Cleaned_Column']

    # Tokenize and get embeddings
    inputs = tokenizer(hindi_text, return_tensors="pt", truncation=True)
    outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()

    # Measure Euclidean distance
    distance = np.linalg.norm(embeddings - cluster_center)

    # Update the 'hin_dis_non_hate' column
    df.at[index, 'hin_dis_non_hate'] = distance

# Show the updated DataFrame
print(df[['Cleaned_Column', 'Hate', 'hin_dis_non_hate']])

df.to_excel('HindiText_hate_non_hate.xlsx', index=False)

from google.colab import files

files.download('HindiText_hate_non_hate.xlsx')

## Adding Distances as additional features for the Embeddings

In [ ]:
import pandas as pd
import torch

# Load your dataset with the distances
df = pd.read_excel('ML_Project_main_hate_fake.xlsx')

# Load the previously saved embeddings
all_embeddings = torch.load("ML_Project_mbert_embeddings.pt")

# Extract the distances from the DataFrame
distances_hate_cls = df['Hindi_Hate_Cluster_Dis'].values.reshape(-1, 1)
distances_Non_hate_cls = df['Hindi_Non-Hate_Cluster_Dis'].values.reshape(-1, 1)
distances_fake_cls = df['Hin_Fake_Cluster_Dis'].values.reshape(-1, 1)
distances_Non_fake_cls = df['Hin_Non-Fake_Cluster_Dis'].values.reshape(-1, 1)

# Add distances as new features to the embeddings
all_embeddings_with_distances = torch.cat([all_embeddings,
                                           torch.tensor(distances_hate_cls, dtype=torch.float),
                                           torch.tensor(distances_Non_hate_cls, dtype=torch.float),
                                           torch.tensor(distances_fake_cls, dtype=torch.float),
                                           torch.tensor(distances_Non_fake_cls, dtype=torch.float)], dim=1)

# Save the modified embeddings with distances as a PyTorch tensor
torch.save(all_embeddings_with_distances, "modified_bert_embeddings_with_distances_772.pt")

print("Modified embeddings with distances saved as modified_bert_embeddings_with_distances.pt")


In [ ]:
embedd = torch.load('modified_bert_embeddings_with_distances_772.pt')
embedd.shape